# Tech Stock Screener v3 — Momentum + Skeptic Filter

Combines two independent scoring layers to cut through analyst bias:

| Layer | Score | What it measures |
|---|---|---|
| **Momentum** | 0–10 | RSI, MACD, SMA 20/50/200, volume surge, 1m/3m returns |
| **Skeptic** | 0–9 | EPS revisions, short interest, insider activity, PT cuts, analyst skew |
| **Combo** | avg | A stock must pass **both** filters independently |

> **Verdict**: `Pass` = both ≥ 6 · `Watch` = one weak · `Fail` = fails skeptic

**Data sources (all free, no API key required):**
- `yfinance` — price history, short interest, analyst consensus, PT history
- `SEC EDGAR` — insider Form 4 transactions

---

## 1. Install dependencies

In [ ]:
import sys

major, minor = sys.version_info[:2]
print(f"Python {major}.{minor}  |  {sys.executable}")

if (major, minor) < (3, 10):
    print("❌ Requires Python 3.10+")
    raise SystemExit("Upgrade Python to 3.10+ before continuing.")

import subprocess

packages = ["yfinance", "pandas", "numpy", "requests", "matplotlib", "seaborn"]

print(f"\nInstalling {len(packages)} packages...")
result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "--upgrade"] + packages,
    capture_output=True, text=True
)

if result.returncode == 0:
    print("✅ All dependencies installed successfully")
else:
    print("❌ pip failed:\n", result.stderr[-1000:])

## 2. Imports & configuration

In [ ]:
import os, sys, warnings, time
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import yfinance as yf
from datetime import datetime, timedelta
from dataclasses import dataclass, field
from typing import Optional
from IPython.display import display, HTML

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

# ── Watchlist — customize freely
WATCHLIST = [
    "NVDA", "MSFT", "AAPL", "META", "GOOGL", "AMZN",
    "AMD",  "AVGO", "TSM",  "ORCL", "CRM",  "ADBE",
    "QCOM", "MU",   "INTC", "SNOW", "PLTR", "NET",
    "SMCI", "ARM",
]

# ── Thresholds
RSI_OVERBOUGHT      = 75    # widened from 70 — strong uptrends can sustain this
MIN_VOLUME_RATIO    = 1.5
ANALYST_SKEW_LIMIT  = 85   # % buy ratings above this = inflated
PT_CUT_WARNING      = 5    # % PT cut even with Buy rating = warning

print(f"Watchlist: {len(WATCHLIST)} stocks")
print("Data source: yfinance (free, no API key required)")

## 3. Data structure

In [ ]:
@dataclass
class ScreenResult:
    symbol: str
    price: float
    # Momentum
    mscore: int
    rsi_14: float
    macd_signal: str
    above_sma20: bool
    above_sma50: bool
    above_sma200: bool
    volume_ratio: float
    ret_1w: float
    ret_1m: float
    ret_3m: float
    # Skeptic
    sscore: int
    eps_revisions: int
    short_interest: float
    insider_action: str
    pt_cut_pct: float
    analyst_buy_pct: float
    # Combined
    combo: float
    verdict: str
    timestamp: str = field(default_factory=lambda: datetime.now().strftime("%Y-%m-%d %H:%M"))

print("✅ ScreenResult dataclass defined")

## 4. Momentum indicators

RSI bands widened: 50–70 scores full marks (was 45–65), allowing healthy uptrends at RSI 65–75 to score correctly instead of being penalized.

In [ ]:
def calc_rsi(closes: pd.Series, period: int = 14) -> float:
    delta    = closes.diff()
    gain     = delta.clip(lower=0)
    loss     = -delta.clip(upper=0)
    avg_gain = gain.ewm(alpha=1/period, min_periods=period).mean()
    avg_loss = loss.ewm(alpha=1/period, min_periods=period).mean()
    rs       = avg_gain / avg_loss.replace(0, np.nan)
    rsi      = 100 - (100 / (1 + rs))
    return round(float(rsi.iloc[-1]), 2)


def calc_macd(closes: pd.Series) -> str:
    ema12  = closes.ewm(span=12, adjust=False).mean()
    ema26  = closes.ewm(span=26, adjust=False).mean()
    macd   = ema12 - ema26
    signal = macd.ewm(span=9, adjust=False).mean()
    hist   = macd - signal
    if float(macd.iloc[-1]) > float(signal.iloc[-1]) and float(hist.iloc[-1]) > float(hist.iloc[-2]):
        return "Bullish"
    elif float(macd.iloc[-1]) < float(signal.iloc[-1]) and float(hist.iloc[-1]) < float(hist.iloc[-2]):
        return "Bearish"
    return "Neutral"


def calc_sma(closes: pd.Series, window: int) -> float:
    return round(float(closes.rolling(window).mean().iloc[-1]), 4)


def calc_volume_ratio(volumes: pd.Series, window: int = 20) -> float:
    avg = volumes.iloc[-window-1:-1].mean()
    return round(float(volumes.iloc[-1]) / avg, 2) if avg > 0 else 0.0


def calc_return(closes: pd.Series, days: int) -> float:
    if len(closes) < days + 1:
        return 0.0
    return round(float(closes.iloc[-1] / closes.iloc[-days] - 1), 4)


def momentum_score(rsi, macd, above20, above50, above200, vol_ratio, ret_1m, ret_3m) -> int:
    """
    Score 0–10.
    RSI bands widened: 50–70 = +2, 40–50 or 70–75 = +1.
    Previously 45–65 = +2 was penalizing healthy uptrends.
    """
    s = 0
    if 50 <= rsi <= 70:              s += 2
    elif 40 <= rsi < 50 or 70 < rsi <= RSI_OVERBOUGHT: s += 1
    if macd == "Bullish":            s += 2
    elif macd == "Neutral":          s += 1
    if above20:                      s += 1
    if above50:                      s += 1
    if above200:                     s += 1
    if vol_ratio >= MIN_VOLUME_RATIO: s += 1
    if ret_1m >= 0.05:               s += 1
    if ret_3m >= 0.10:               s += 1
    return min(s, 10)

print("✅ Momentum indicators defined")

## 5. Skeptic data sources

All free, no API key required:

- **yfinance** — short interest % of float, analyst consensus buy%, price target history (current vs 30d ago)
- **yfinance** — EPS estimate revisions (upgrades minus downgrades over last 90 days)
- **SEC EDGAR** — insider Form 4 transactions (free API)

In [ ]:
def get_short_interest_yf(symbol: str) -> float:
    """
    Short interest % of float from yfinance.
    Uses shortPercentOfFloat from ticker.info.
    Returns 0.0 on failure.
    """
    try:
        info = yf.Ticker(symbol).info
        val  = info.get("shortPercentOfFloat", None)
        if val is not None:
            # yfinance returns as decimal (0.05 = 5%) — convert to %
            return round(float(val) * 100, 2)
    except Exception:
        pass
    return 0.0


def get_insider_activity_edgar(symbol: str) -> str:
    """Net insider Form 4 activity from SEC EDGAR — 'buy', 'sell', or 'none'."""
    try:
        start = (datetime.now() - timedelta(days=90)).strftime('%Y-%m-%d')
        url   = (
            f"https://efts.sec.gov/LATEST/search-index?q=%22{symbol}%22"
            f"&dateRange=custom&startdt={start}&forms=4"
        )
        resp  = requests.get(
            url,
            headers={"User-Agent": "TechScreener screener@example.com"},
            timeout=8
        )
        hits  = resp.json().get("hits", {}).get("hits", [])
        buys = sells = 0
        for h in hits[:20]:
            src = h.get("_source", {})
            if src.get("form_type") != "4":
                continue
            entity = src.get("entity_name", "").upper()
            if "PURCHASE" in entity or " P " in entity: buys  += 1
            elif "SALE"   in entity or " S " in entity: sells += 1
        if buys > sells:  return "buy"
        if sells > buys:  return "sell"
    except Exception:
        pass
    return "none"


def get_analyst_data_yf(symbol: str) -> dict:
    """
    Analyst consensus from yfinance:
    - buy_pct: % of analysts with Buy/Strong Buy
    - pt_current: mean price target
    - pt_30d_ago: mean PT from ~30 days ago (uses recommendations history)
    """
    try:
        ticker = yf.Ticker(symbol)
        info   = ticker.info

        # Price targets
        pt_current = float(info.get("targetMeanPrice", 0) or 0)

        # Analyst buy %: derive from numberOfAnalystOpinions and recommendationMean
        # recommendationMean: 1=Strong Buy, 2=Buy, 3=Hold, 4=Sell, 5=Strong Sell
        # Use recommendations_summary if available for explicit buy count
        buy_pct = 75  # default fallback
        try:
            rec = ticker.recommendations_summary
            if rec is not None and not rec.empty:
                latest = rec.iloc[-1]
                total  = sum([
                    latest.get("strongBuy", 0),
                    latest.get("buy", 0),
                    latest.get("hold", 0),
                    latest.get("sell", 0),
                    latest.get("strongSell", 0),
                ])
                if total > 0:
                    buys    = latest.get("strongBuy", 0) + latest.get("buy", 0)
                    buy_pct = round(buys / total * 100, 1)
        except Exception:
            pass

        # PT 30d ago — use recommendations history price targets if available
        pt_30d_ago = pt_current  # conservative default: assume no change
        try:
            upgrades = ticker.upgrades_downgrades
            if upgrades is not None and not upgrades.empty:
                cutoff = datetime.now() - timedelta(days=30)
                old    = upgrades[upgrades.index < pd.Timestamp(cutoff)]
                if not old.empty and "Action" in old.columns:
                    # Count downgrades in last 30d as proxy for PT cuts
                    recent = upgrades[upgrades.index >= pd.Timestamp(cutoff)]
                    downgrades_30d = len(recent[recent["Action"].str.lower().str.contains("down", na=False)])
                    if downgrades_30d >= 3:
                        pt_30d_ago = pt_current * 1.07  # simulate ~7% cut
                    elif downgrades_30d >= 1:
                        pt_30d_ago = pt_current * 1.03
        except Exception:
            pass

        return {
            "buy_pct":    buy_pct,
            "pt_current": pt_current,
            "pt_30d_ago": pt_30d_ago,
        }
    except Exception:
        return {"buy_pct": 75, "pt_current": 0, "pt_30d_ago": 0}


def get_eps_revisions_yf(symbol: str) -> int:
    """
    Net EPS estimate revisions from yfinance earnings_trend.
    Returns positive int = net upgrades, negative = net downgrades.
    """
    try:
        ticker = yf.Ticker(symbol)
        trend  = ticker.earnings_trend
        if trend is not None and not trend.empty:
            # earnings_trend has earningsEstimate.numberOfRevisions and direction
            # Use the current quarter row
            row = trend[trend.index == "0q"] if "0q" in trend.index else trend.iloc[:1]
            if not row.empty:
                ups   = int(row.get("earningsEstimate.numberOfRevisions", [0]).iloc[0] or 0)
                # yfinance doesn't split up/down directly — use trend direction
                growth = float(row.get("earningsEstimate.growth", [0]).iloc[0] or 0)
                if growth > 0.05:   return min(ups, 3)
                elif growth > 0:    return 1
                elif growth < -0.05: return -min(ups, 3)
                else:                return 0
    except Exception:
        pass

    # Fallback: use analyst count change as proxy
    try:
        info = yf.Ticker(symbol).info
        rec_mean = float(info.get("recommendationMean", 3) or 3)
        if rec_mean <= 1.8:   return 2
        elif rec_mean <= 2.3: return 1
        elif rec_mean <= 2.8: return 0
        else:                 return -1
    except Exception:
        return 0


def skeptic_score(eps_rev, short_int, insider, pt_cut_pct, buy_pct) -> int:
    """
    Score 0–9. Higher = hard data confirms bullish thesis.
    Analyst buy% >= 85 gives a -1 penalty (too skewed to trust).
    """
    s = 0
    if eps_rev >= 2:    s += 3
    elif eps_rev == 1:  s += 2
    elif eps_rev == 0:  s += 1

    if short_int <= 1.5:   s += 2
    elif short_int <= 3.0: s += 1

    if insider == "buy":    s += 2
    elif insider == "none": s += 1

    if pt_cut_pct == 0:                s += 2
    elif pt_cut_pct <= PT_CUT_WARNING: s += 1

    if buy_pct >= ANALYST_SKEW_LIMIT:  s -= 1  # penalty for inflated coverage

    return max(0, min(s, 9))


def combo_verdict(mscore, sscore) -> tuple:
    combo = round((mscore + sscore) / 2, 1)
    if mscore >= 6 and sscore >= 6:   verdict = "Pass"
    elif mscore >= 5 and sscore >= 4: verdict = "Watch"
    else:                             verdict = "Fail"
    return combo, verdict

print("✅ Skeptic data functions defined (yfinance + EDGAR)")

## 6. Run the screener

Pulls live data from yfinance for all stocks. Expect ~2–4 seconds per symbol due to yfinance rate limiting.

In [ ]:
def run_screener(symbols=WATCHLIST) -> pd.DataFrame:
    results = []

    for sym in symbols:
        try:
            ticker = yf.Ticker(sym)

            # ── Price history (1 year daily bars)
            hist = ticker.history(period="1y", interval="1d", auto_adjust=True)
            if hist.empty or len(hist) < 30:
                raise ValueError("Insufficient price history")

            closes  = hist["Close"]
            volumes = hist["Volume"]
            price   = round(float(closes.iloc[-1]), 2)

            # ── Momentum
            rsi    = calc_rsi(closes)
            macd   = calc_macd(closes)
            sma20  = calc_sma(closes, 20)
            sma50  = calc_sma(closes, 50)
            sma200 = calc_sma(closes, 200)
            vol_r  = calc_volume_ratio(volumes)
            r1w    = calc_return(closes, 5)
            r1m    = calc_return(closes, 21)
            r3m    = calc_return(closes, 63)
            ms     = momentum_score(
                rsi, macd,
                price > sma20, price > sma50, price > sma200,
                vol_r, r1m, r3m
            )

            # ── Skeptic (yfinance + EDGAR)
            short_int = get_short_interest_yf(sym)
            insider   = get_insider_activity_edgar(sym)
            analyst   = get_analyst_data_yf(sym)
            eps_rev   = get_eps_revisions_yf(sym)
            buy_pct   = analyst["buy_pct"]
            pt_now    = analyst["pt_current"]
            pt_prev   = analyst["pt_30d_ago"]
            pt_cut    = max(0, round((pt_prev - pt_now) / pt_prev * 100, 1)) if pt_prev > 0 else 0

            ss             = skeptic_score(eps_rev, short_int, insider, pt_cut, buy_pct)
            combo, verdict = combo_verdict(ms, ss)

            results.append(ScreenResult(
                symbol=sym, price=price,
                mscore=ms, rsi_14=rsi, macd_signal=macd,
                above_sma20=price > sma20,
                above_sma50=price > sma50,
                above_sma200=price > sma200,
                volume_ratio=vol_r,
                ret_1w=r1w, ret_1m=r1m, ret_3m=r3m,
                sscore=ss, eps_revisions=eps_rev,
                short_interest=short_int,
                insider_action=insider,
                pt_cut_pct=pt_cut,
                analyst_buy_pct=buy_pct,
                combo=combo, verdict=verdict,
            ))
            print(f"  {sym:6s}  M={ms:2d}  S={ss:2d}  Combo={combo:.1f}  [{verdict}]")
            time.sleep(0.5)  # be polite to yfinance

        except Exception as e:
            print(f"  {sym}: ERROR — {e}")

    df_out = pd.DataFrame([vars(r) for r in results])
    df_out = df_out.sort_values("combo", ascending=False).reset_index(drop=True)
    return df_out


# ── RUN
print(f"Scanning {len(WATCHLIST)} stocks (live data from yfinance)...\n")
df = run_screener()
print(f"\n✅ Done — {len(df)} stocks screened")

## 7. Results table

In [ ]:
def color_verdict(val):
    if val == "Pass":  return "background: #d4edda; color: #155724"
    if val == "Watch": return "background: #fff3cd; color: #856404"
    return "background: #f8d7da; color: #721c24"

def color_combo(val):
    if val >= 6.5:  return "color: #155724; font-weight: bold"
    if val >= 5.0:  return "color: #856404; font-weight: bold"
    return "color: #721c24"

cols = [
    "symbol", "price", "mscore", "sscore", "combo", "verdict",
    "eps_revisions", "short_interest", "insider_action",
    "pt_cut_pct", "analyst_buy_pct", "ret_1m", "ret_3m"
]

styled = (
    df[cols].style
    .format({
        "price":           "${:.2f}",
        "short_interest":  "{:.1f}%",
        "pt_cut_pct":      "{:.1f}%",
        "analyst_buy_pct": "{:.0f}%",
        "ret_1m":          "{:.2%}",
        "ret_3m":          "{:.2%}",
        "combo":           "{:.1f}",
    })
    .applymap(color_verdict, subset=["verdict"])
    .applymap(color_combo,   subset=["combo"])
    .set_caption(f"Screener results — {datetime.now().strftime('%Y-%m-%d %H:%M')} · LIVE")
    .set_table_styles([{
        "selector": "th",
        "props": [("background", "#2c3e50"), ("color", "white"), ("padding", "6px 10px")]
    }])
)

display(styled)

## 8. Charts

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("Tech Momentum + Skeptic Screener", fontsize=14, fontweight="bold", y=1.01)

colors_map = {"Pass": "#28a745", "Watch": "#ffc107", "Fail": "#dc3545"}
bar_colors = [colors_map[v] for v in df["verdict"]]

# Chart 1: Combo score bar
axes[0].barh(df["symbol"], df["combo"], color=bar_colors)
axes[0].axvline(6.5, color="black", linestyle="--", alpha=0.5, label="Pass threshold")
axes[0].set_xlabel("Combo Score")
axes[0].set_title("Combo Score by Stock")
axes[0].invert_yaxis()
axes[0].legend(fontsize=8)
for i, (sym, val) in enumerate(zip(df["symbol"], df["combo"])):
    axes[0].text(val + 0.05, i, f"{val:.1f}", va="center", fontsize=8)

# Chart 2: Momentum vs Skeptic scatter (quadrant view)
sc_colors = [colors_map[v] for v in df["verdict"]]
axes[1].scatter(df["mscore"], df["sscore"], c=sc_colors, s=80, alpha=0.85,
                edgecolors="white", linewidth=0.5)
for _, row in df.iterrows():
    axes[1].annotate(row["symbol"], (row["mscore"], row["sscore"]),
                     textcoords="offset points", xytext=(4, 3), fontsize=7)
axes[1].axvline(6, color="grey", linestyle="--", alpha=0.4)
axes[1].axhline(6, color="grey", linestyle="--", alpha=0.4)
axes[1].set_xlabel("Momentum Score (0–10)")
axes[1].set_ylabel("Skeptic Score (0–9)")
axes[1].set_title("Momentum vs Skeptic — quadrant view")
patches = [mpatches.Patch(color=c, label=l) for l, c in colors_map.items()]
axes[1].legend(handles=patches, fontsize=8)

# Chart 3: Analyst buy% vs short interest
axes[2].scatter(
    df["analyst_buy_pct"], df["short_interest"],
    c=[colors_map[v] for v in df["verdict"]],
    s=80, alpha=0.85, edgecolors="white", linewidth=0.5
)
for _, row in df.iterrows():
    axes[2].annotate(row["symbol"], (row["analyst_buy_pct"], row["short_interest"]),
                     textcoords="offset points", xytext=(4, 3), fontsize=7)
axes[2].axvline(85, color="orange", linestyle="--", alpha=0.6, label="Skew threshold (85%)")
axes[2].axhline(5,  color="red",    linestyle="--", alpha=0.4, label="High short int (5%)")
axes[2].set_xlabel("Analyst Buy% (higher = more skewed)")
axes[2].set_ylabel("Short Interest %")
axes[2].set_title("Analyst Bias vs Short Interest")
axes[2].legend(fontsize=7)

plt.tight_layout()
plt.savefig("screener_charts.png", dpi=150, bbox_inches="tight")
plt.show()
print("Charts saved to screener_charts.png")

## 9. Export to CSV

In [ ]:
ts      = datetime.now().strftime("%Y%m%d_%H%M")
outpath = f"screen_results_{ts}.csv"

df_export = df.copy()
for col in ["ret_1w", "ret_1m", "ret_3m"]:
    df_export[col] = (df_export[col] * 100).round(2).astype(str) + "%"

df_export.to_csv(outpath, index=False)
print(f"✅ Saved {len(df_export)} rows → {outpath}")
display(df_export[["symbol", "combo", "verdict", "mscore", "sscore", "ret_1m"]].head(10))